In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]

SCHEMA = "skypoints"

SILVER_TABLE = f"{catalog}.{SCHEMA}.silver_member"

GOLD_MASTER = f"{catalog}.{SCHEMA}.gold_member_master"
GOLD_IND = f"{catalog}.{SCHEMA}.gold_member_ind"
GOLD_USA = f"{catalog}.{SCHEMA}.gold_member_usa"
GOLD_AUS = f"{catalog}.{SCHEMA}.gold_member_aus"

Read the data from silver layer

In [0]:
silver_df = spark.table(SILVER_TABLE)

In [0]:
print("Silver record count:", silver_df.count())

Implement latest-record-wins logic

In [0]:
display(silver_df)

In [0]:
window_spec = (
    Window
    .partitionBy("SOURCE_COUNTRY", "MEMBER_ID")
    .orderBy(
        F.col("LAST_UPDATED_DATE").desc_nulls_last()
    )
)

gold_df = (
    silver_df
    .withColumn("ROW_NUM", F.row_number().over(window_spec))
    .filter(F.col("ROW_NUM")  == 1)
    .drop("ROW_NUM")
)

Created gold layer for master customer

In [0]:
gold_df = gold_df.withColumnRenamed(
    "MEMBER_ID",
    "SOURCE_MEMBER_ID"
)

In [0]:
gold_df = gold_df.withColumn(
    "MEMBER_KEY",
    F.concat_ws(
        "_",
        F.col("SOURCE_COUNTRY"),
        F.col("SOURCE_MEMBER_ID")
    )
)

In [0]:
(
    gold_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_MASTER)
)

In [0]:
display(spark.table(GOLD_MASTER))

Split members by country

In [0]:
ind_df = (gold_df.filter(F.col("SOURCE_COUNTRY") == "IND"))

(
    ind_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(GOLD_IND)
)

usa_df = gold_df.filter(F.col("SOURCE_COUNTRY") == "USA")

(
    usa_df.write 
    .format("delta") 
    .mode("overwrite") 
    .option("overwriteSchema", "true") 
    .saveAsTable(GOLD_USA)
)

aus_df = gold_df.filter(F.col("SOURCE_COUNTRY") == "AUS")

(
    aus_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_AUS)
)

In [0]:
# verify that the country split is correct.
print("Gold Master:", spark.table(GOLD_MASTER).count())
print("India:", spark.table(GOLD_IND).count())
print("USA:", spark.table(GOLD_USA).count())
print("Australia:", spark.table(GOLD_AUS).count())

In [0]:
#Check country-wise counts
(
    spark.table(GOLD_MASTER)
    .groupBy("SOURCE_COUNTRY")
    .count()
    .orderBy("SOURCE_COUNTRY")
).show()

In [0]:
# Check for duplicate member keys within each country
(
    spark.table(GOLD_MASTER)
    .groupBy("SOURCE_COUNTRY", "SOURCE_MEMBER_ID")
    .count()
    .filter(F.col("count") > 1)
).show()